<a href="https://colab.research.google.com/github/wjdwogns2873-web/deep-learning-study/blob/main/Kaggle_Study/09_Rice_Image.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("muratkokludataset/rice-image-dataset")

print("Path to dataset files:", path)

100%|██████████| 219M/219M [00:01<00:00, 130MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/muratkokludataset/rice-image-dataset/versions/1


In [ ]:
!ls -al /root/.cache/kagglehub/datasets/muratkokludataset/rice-image-dataset/versions/1/Rice_Image_Dataset
# !ls -al /kaggle/input/rice-image-dataset/Rice_Image_Dataset/

total 2940
drwxr-xr-x 7 root root   4096 May 31 08:39 .
drwxr-xr-x 3 root root   4096 May 31 08:39 ..
drwxr-xr-x 2 root root 602112 May 31 08:39 Arborio
drwxr-xr-x 2 root root 557056 May 31 08:39 Basmati
drwxr-xr-x 2 root root 593920 May 31 08:39 Ipsala
drwxr-xr-x 2 root root 598016 May 31 08:39 Jasmine
drwxr-xr-x 2 root root 626688 May 31 08:39 Karacadag
-rw-r--r-- 1 root root   3053 May 31 08:39 Rice_Citation_Request.txt


```
CSV(DataFrame) 데이터 빌딩
```

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold

# 1. 이미지 경로 및 라벨 수집
DATA_DIR = '/root/.cache/kagglehub/datasets/muratkokludataset/rice-image-dataset/versions/1/Rice_Image_Dataset'
# DATA_DIR = '/kaggle/input/rice-image-dataset/Rice_Image_Dataset/'
categories = ['Arborio', 'Basmati', 'Ipsala', 'Jasmine', 'Karacadag']

data = []
for category in categories:
    folder_path = os.path.join(DATA_DIR, category)
    for img_name in os.listdir(folder_path):
        if img_name.endswith(('.jpg', '.jpeg', '.png')):
            data.append({
                'image_path': os.path.join(folder_path, img_name),
                'label_name': category
            })

df = pd.DataFrame(data)

# 라벨 인코딩(문자열 -> 숫자 변환)
label_map = {name: i for i, name in enumerate(categories)}
df['label'] = df['label_name'].map(label_map)

# 2. 15%를 완전히 격리된 Test Set으로 분리 (Stratified로 비율 유지)
train_val_df, test_df = train_test_split(
    df, test_size=0.15, random_state=42, stratify=df['label']
)
train_val_df = train_val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# 3. 남은 85% 데이터에 5-Fold 인덱스 부여 (StratifiedKFold)
train_val_df['fold'] = -1
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_index, val_idx) in enumerate(skf.split(train_val_df, train_val_df['label'])):
    train_val_df.loc[val_idx, 'fold'] = fold

print(f"Train/Val 데이터 개수: {len(train_val_df)}, Test 데이터 개수: {len(test_df)}")

Train/Val 데이터 개수: 63750, Test 데이터 개수: 11250


```
os.listdir()는 지정한 디렉토리 안에 있는 모든 파일과 폴더의 이름을 리스트로 반환합니다.

endswith()는 문자열이 특정 문자로 끝나는지 확인하여 True 또는 False를 반환하는 메서드입니다.

<예제 코드1>
text = "report.xlsx"

print(text.endswith(".xlsx")) # True
print(text.endswith(".docx")) # False

<예제 코드2>
import os
files = ["img1.jpg", "notes.txt", "photo.png", "script.py"]

# .jpg 또는 .png로 끝나는 파일만 골라내기
image_extensions = (".jpg", ".jpeg", ".png")
images = [f for f in files if f.lower().endswith(image_extensions)]

print(images) # ['img1.jpg', 'photo.png']
```

```
PyTorch Dataset 클래스 정의
```

In [ ]:
import cv2
import torch
from torch.utils.data import Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2

class RiceDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = cv2.imread(row['image_path'])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        label = int(row['label'])

        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']

        return image, torch.tensor(label, dtype=torch.long)

train_transform = A.Compose([
    A.Resize(128, 128),
    A.HorizontalFlip(p=0.5), # 쌀알 뒤집기 정도의 가벼운 증강
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_test_transform = A.Compose([
    A.Resize(128, 128),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

```
Test 및 Validation을 포함한 올인원 루프
핵심은 각 폴드의 베스트 모델 가중치를 저장하고, 마지막에 그 가중치들을 다 불러와서 Test 데이터를 앙상블(Soft Voting)하는 하름입니다.
```

In [ ]:
from torch.utils.data import DataLoader
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
epochs = 3
BATCH_SIZE = 128

# 최종 Test 예측값들을 누적할 행렬 (데이터수 x 클래스수)
test_dataset = RiceDataset(test_df, transform=val_test_transform)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
total_test_preds = torch.zeros((len(test_df), len(categories))).to(device)

for fold in range(5):
    # 데이터 분할 (조건절 필터링)
    train_data = train_val_df[train_val_df['fold'] != fold].reset_index(drop=True)
    val_data = train_val_df[train_val_df['fold'] == fold].reset_index(drop=True)

    train_loader = DataLoader(RiceDataset(train_data, train_transform), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(RiceDataset(val_data, val_test_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    # 모델 정의 (가벼운 resnet18이나 efficientnet_b0 추천)
    model = timm.create_model('resnet18', pretrained=True, num_classes=5).to(device)
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

    best_val_loss = float('inf')

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        train_correct = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            # 정확한 통계를 위해 배치 사이즈(images.size(0))를 곱해서 누적
            train_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)
            train_correct += (preds == labels).sum().item()

        epoch_train_loss = train_loss / len(train_data)
        epoch_train_acc = train_correct / len(train_data)
        print(f"Epoch {epoch+1} | Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc * 100:.2f}%")

        model.eval()
        val_loss = 0.0
        correct = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)
                preds = outputs.argmax(dim=1)
                correct += (preds == labels).sum().item()

        epoch_val_loss = val_loss / len(val_data)
        epoch_val_acc = correct / len(val_data)
        print(f"Epoch {epoch+1} - Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.4f}")

        # Best 가중치 저장
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            torch.save(model.state_dict(), f"best_model_fold{fold}.pth")

    # [Fold 종료 후 Test Data Inference]
    print(f"--> Load Best Model of Fold {fold} and Predict Test Data")
    best_model = timm.create_model('resnet18', pretrained=False, num_classes=5).to(device)
    best_model.load_state_dict(torch.load(f"best_model_fold{fold}.pth"))
    best_model.eval()

    fold_test_preds = []
    with torch.no_grad():
        for images, _ in test_loader:
            images = images.to(device)
            outputs = best_model(images)
            # Softmax를 통해 확률값으로 변환하여 누적
            probs = torch.softmax(outputs, dim=1)
            fold_test_preds.append(probs)

    # 이번 폴드의 예측치를 마스터 행렬에 합산 (앙상블 준비)
    total_test_preds += torch.cat(fold_test_preds, dim=0)

# 5개 폴드의 예측치 평균 (Sort Voting 앙상블)
final_test_preds = total_test_preds / 5
final_classes = final_test_preds.argmax(dim=1).cpu().numpy()

# 최종 결과물 매핑 및 제출(Submission) 준비
test_df['predicted_label'] = final_classes
test_df.to_csv('submission.csv', index=False)

Epoch 1 | Train Loss: 0.0433 | Train Acc: 98.95%
Epoch 1 - Val Loss: 0.0088, Val Acc: 0.9970
Epoch 2 | Train Loss: 0.0103 | Train Acc: 99.69%
Epoch 2 - Val Loss: 0.0058, Val Acc: 0.9983
Epoch 3 | Train Loss: 0.0063 | Train Acc: 99.80%
Epoch 3 - Val Loss: 0.0430, Val Acc: 0.9869
--> Load Best Model of Fold 0 and Predict Test Data
Epoch 1 | Train Loss: 0.0443 | Train Acc: 98.92%
Epoch 1 - Val Loss: 0.0055, Val Acc: 0.9983
Epoch 2 | Train Loss: 0.0122 | Train Acc: 99.65%
Epoch 2 - Val Loss: 0.1039, Val Acc: 0.9735
Epoch 3 | Train Loss: 0.0091 | Train Acc: 99.73%
Epoch 3 - Val Loss: 0.0194, Val Acc: 0.9935
--> Load Best Model of Fold 1 and Predict Test Data
Epoch 1 | Train Loss: 0.0446 | Train Acc: 98.93%
Epoch 1 - Val Loss: 0.0494, Val Acc: 0.9840
Epoch 2 | Train Loss: 0.0102 | Train Acc: 99.71%
Epoch 2 - Val Loss: 0.0042, Val Acc: 0.9990
Epoch 3 | Train Loss: 0.0083 | Train Acc: 99.73%
Epoch 3 - Val Loss: 0.0059, Val Acc: 0.9983
--> Load Best Model of Fold 2 and Predict Test Data
Epoch 1